In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


In [0]:
import mlflow
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import pandas as pd

In [0]:
# Read gold table
df = spark.table("workspace.default.gold_diabetic").toPandas()

print(f"Dataset shape: {df.shape}")
print(f"Readmission rate: {df['readmitted_flag'].mean():.2%}")

Dataset shape: (101763, 14)
Readmission rate: 11.16%


In [0]:
# Separate features and target
X = df.drop(columns=["readmitted_flag"])
y = df["readmitted_flag"]

# Split 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")
print(f"Readmission rate in train: {y_train.mean():.2%}")
print(f"Readmission rate in test: {y_test.mean():.2%}")

Train size: 81410
Test size: 20353
Readmission rate in train: 11.16%
Readmission rate in test: 11.16%


In [0]:
mlflow.set_experiment("/healthlake-readmission")

with mlflow.start_run(run_name="xgboost_baseline"):
    
    # Model parameters
    params = {
        "n_estimators": 200,
        "max_depth": 5,
        "learning_rate": 0.1,
        "scale_pos_weight": 8,
        "random_state": 42
    }
    
    # Train
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    
    # Log to MLflow with signature
    mlflow.log_params(params)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("f1_score", f1)
    mlflow.sklearn.log_model(
        model,
        name="model",
        input_example=X_train.iloc[:5]
    )
    
    print(f"AUC-ROC: {auc:.4f}")
    print(f"F1-Score: {f1:.4f}")

🔗 View Logged Model at: https://dbc-f2aaf8c4-fea6.cloud.databricks.com/ml/experiments/866010618732186/models/m-e4e0958c198b4360b66411b5606d9441?o=7474649208316381
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: Us

AUC-ROC: 0.6340
F1-Score: 0.2520


In [0]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.67      0.77     18082
           1       0.17      0.52      0.25      2271

    accuracy                           0.65     20353
   macro avg       0.54      0.60      0.51     20353
weighted avg       0.83      0.65      0.72     20353



In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the last run id
runs = mlflow.search_runs(experiment_names=["/healthlake-readmission"])
run_id = runs.iloc[0]["run_id"]

# Register model
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri, "healthlake-readmission-model")

print(f"Model registered successfully!")
print(f"Run ID: {run_id}")

Registered model 'healthlake-readmission-model' already exists. Creating a new version of this model...
2026/06/02 04:38:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 8b49592cfd234c478f93896505df9696 has no artifacts at artifact path 'model', registering model based on models:/m-e4e0958c198b4360b66411b5606d9441 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

Model registered successfully!
Run ID: 8b49592cfd234c478f93896505df9696


🔗 Created version '1' of model 'workspace.default.healthlake-readmission-model': https://dbc-f2aaf8c4-fea6.cloud.databricks.com/explore/data/models/workspace/default/healthlake-readmission-model/version/1?o=7474649208316381
